# GMP Utility Maintenance Screening
**Author:** Hanson Tawiah

Cleaned from the original ArcGIS Pro analysis notebooks. This notebook classifies existing maintenance scores, performs a NumPy PCA anomaly screen, and optionally exports a saved dashboard configuration.

## Requirements and scope
Run inside an ArcGIS Pro notebook with an open project containing `GMP_StudyArea_Poles` and, for inventory reporting, `GMP_StudyArea_Distribution_Lines`. ArcPy and NumPy are required; the optional export uses the ArcGIS API for Python and your existing Pro sign-in.

The pole layer must already contain `Pole_Age`, `Priority_Score`, `POLEHEIGHT`, `TRANSFRMR`, `RISERS`, and `GUYS`. **The original age and score calculation code was not present in the supplied notebooks. This notebook does not recreate those calculations or the raw-data preparation.**

Classification and anomaly cells update the local pole dataset. Run on a working copy. Changes to local data do not automatically update published hosted layers.

## Recorded project results
The original saved outputs reported 11,133 poles: High 1,299; Medium 4,348; Low 5,479; Unknown 7. PCA screened 11,126 poles, flagged 561, and excluded 7 with missing age. There were 31 distribution line feature records, which need not represent 31 individual spans.
These are historical results, not outputs from executing this cleaned notebook.


In [ ]:
import arcpy
import numpy as np
import csv
from pathlib import Path
from collections import Counter

aprx = arcpy.mp.ArcGISProject("CURRENT")
matches = [layer for map_ in aprx.listMaps() for layer in map_.listLayers()
           if layer.name == "GMP_StudyArea_Poles"]
if not matches:
    raise ValueError("Add GMP_StudyArea_Poles to the project first.")
pole_layer = matches[0]
fc = pole_layer.dataSource
required = {"Pole_Age", "Priority_Score", "POLEHEIGHT", "TRANSFRMR", "RISERS", "GUYS"}
missing = required - {f.name for f in arcpy.ListFields(fc)}
if missing:
    raise ValueError(f"Missing required input fields: {sorted(missing)}")
print("Pole records:", arcpy.management.GetCount(fc)[0])


## 1. Classify existing maintenance scores
Use the original thresholds: High ≥70; Medium ≥40 and <70; Low <40; Unknown for null scores. These thresholds are a project screening rule, not calibrated failure probabilities.


In [ ]:
if "Priority_Class" not in {f.name for f in arcpy.ListFields(fc)}:
    arcpy.management.AddField(fc, "Priority_Class", "TEXT", field_length=12)

counts = Counter()
with arcpy.da.UpdateCursor(fc, ["Priority_Score", "Priority_Class"]) as cursor:
    for row in cursor:
        score = row[0]
        category = ("Unknown" if score is None else
                    "High" if score >= 70 else
                    "Medium" if score >= 40 else "Low")
        row[1] = category
        cursor.updateRow(row)
        counts[category] += 1

output = Path(aprx.homeFolder) / "GMP_Priority_Summary.csv"
with output.open("w", newline="", encoding="utf-8") as file:
    writer = csv.writer(file)
    writer.writerow(["Priority_Class", "Pole_Count"])
    for category in ("High", "Medium", "Low", "Unknown"):
        writer.writerow([category, counts[category]])
print(dict(counts))
print("Summary saved:", output)


## 2. PCA anomaly screening
The working original method median-fills missing or nonpositive heights, encodes `Y` as 1 and all other attachment values as 0, standardizes five inputs, and uses the first three principal components. Its score is the sum of squared component coordinates scaled by component standard deviations. Scores at or above the dataset's 95th percentile are flagged; ties can make the flagged share exceed 5%.

Missing attachment values are treated as 0 by the original code, which is a data-quality limitation. The percentile is a chosen review threshold, not a learned failure rate. The method identifies unusual records, not confirmed damage or predicted failures.


In [ ]:
fields = ["OID@", "Pole_Age", "POLEHEIGHT", "TRANSFRMR", "RISERS", "GUYS"]
records = list(arcpy.da.SearchCursor(fc, fields))
valid = [r for r in records if r[1] is not None]

if len(valid) < 2:
    raise ValueError("At least two poles with known age are required.")

heights = [float(r[2]) for r in valid
           if r[2] is not None and float(r[2]) > 0]
if not heights:
    raise ValueError("No positive pole heights are available for median filling.")
height_fill = float(np.median(heights))

X = np.array([
    [float(r[1]),
     float(r[2]) if r[2] is not None and float(r[2]) > 0 else height_fill,
     int(str(r[3]).upper() == "Y"),
     int(str(r[4]).upper() == "Y"),
     int(str(r[5]).upper() == "Y")]
    for r in valid
])

if not np.isfinite(X).all():
    raise ValueError("Input attributes contain nonfinite numeric values.")

# Standardize fields, then find unusual combinations using PCA.
Z = (X - X.mean(axis=0)) / np.maximum(X.std(axis=0), 0.001)
_, singular, components = np.linalg.svd(Z, full_matrices=False)
pc = Z @ components[:3].T
scores = np.sum(
    (pc / np.maximum(singular[:3] / np.sqrt(len(Z) - 1), 0.001)) ** 2,
    axis=1
)
cutoff = np.percentile(scores, 95)
results = {r[0]: int(score >= cutoff)
           for r, score in zip(valid, scores)}

if "ML_Anomaly" not in [f.name for f in arcpy.ListFields(fc)]:
    arcpy.management.AddField(fc, "ML_Anomaly", "SHORT")

with arcpy.da.UpdateCursor(fc, ["OID@", "ML_Anomaly"]) as cursor:
    for row in cursor:
        row[1] = results.get(row[0])
        cursor.updateRow(row)

print(f"Screened: {len(valid):,}")
print(f"Flagged for review: {sum(results.values()):,}")
print(f"Unknown age: {len(records) - len(valid):,}")

## 3. Optional dashboard configuration export
Exports the currently saved online dashboard JSON using your Pro session. This is a configuration backup: it depends on the web map and hosted services and does not export their underlying data. It does not build or modify a dashboard.


In [ ]:
from arcgis.gis import GIS
import json

gis = GIS("pro")
dashboard_id = "f6d21e3bc9104bdd872f2ec5305ae4cd"
item = gis.content.get(dashboard_id)
if item is None:
    raise ValueError("Dashboard is unavailable to the current account.")
config = item.get_data()
if not isinstance(config, dict):
    raise ValueError("Dashboard did not return a JSON configuration.")
folder = Path(aprx.homeFolder) / "GMP_Dashboard_Export"
folder.mkdir(exist_ok=True)
file = folder / "dashboard.json"
file.write_text(json.dumps(config, indent=2), encoding="utf-8")
print("Export saved:", file)


## Interpretation and validation limits
Inspection and failure records are needed to assess whether the screening identifies real maintenance needs. No predictive accuracy, maintenance savings, or operational improvement was measured. No distribution line risk model or 3D digital twin is implemented here.

This cleaned version removes failed Isolation Forest imports, renderer troubleshooting, duplicate JSON dumps, personal output paths, and dashboard layout mutation cells. PCA scoring and classification thresholds are preserved; input guards were added. Code cells were syntax-checked but have not been rerun against the ArcGIS Pro dataset in this environment.

[Live dashboard](https://gis-siue.maps.arcgis.com/apps/dashboards/f6d21e3bc9104bdd872f2ec5305ae4cd)
